# Democratic Sweep and Guardian Policy Indices: Main Results

This notebook reads the completed download, sentiment-index, correlation, and out-of-sample backtest outputs. It does not download data again, retrain LDA, or call DeepSeek. Polymarket prices are proxies for market-implied probabilities; sentiment and party stances presented in Guardian text do not represent all U.S. voters.

In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, HTML

ROOT = Path.cwd()
DATA = ROOT / 'data'

TOPIC_NAMES = {
    0: 'AI, Datacenters, and Digital Governance',
    1: 'Infrastructure, Public Lands, and Federal Investment',
    2: 'Trade, Public Health, and Legal Rights',
    3: 'Iran War, Oil, and Energy Security',
    4: 'Immigration, Public Funding, and Household Economy',
    5: 'Defense Spending and Military Security',
}
INDEX_NAMES = {
    'topic_sentiment': 'Topic sentiment',
    'democrat_stance': 'Democratic stance',
    'republican_stance': 'Republican stance',
    'topic_attention': 'Topic attention',
}

display(HTML('''<style>
.jp-RenderedHTMLCommon table {font-size: 13px;}
.jp-RenderedHTMLCommon th {text-align: center !important;}
.jp-RenderedHTMLCommon td {padding: 6px 10px !important;}
</style>'''))

In [2]:
daily = pd.read_csv(DATA / 'polymarket' / 'daily_sweep_probability.csv', encoding='utf-8-sig', parse_dates=['date'])
coverage = pd.read_csv(DATA / 'sweep_analysis' / 'alignment_coverage_summary.csv', encoding='utf-8-sig')
corr = pd.read_csv(DATA / 'sweep_analysis' / 'correlation_results.csv', encoding='utf-8-sig')
metrics = pd.read_csv(DATA / 'sweep_analysis' / 'walk_forward_metrics.csv', encoding='utf-8-sig')
quality = json.loads((DATA / 'polymarket' / 'download_quality_report.json').read_text(encoding='utf-8'))
validation = json.loads((DATA / 'sweep_analysis' / 'analysis_validation_report.json').read_text(encoding='utf-8'))

assert len(daily) == 92
assert daily['quality_flag'].eq('ok').all()
assert len(coverage) == 24
assert validation['validation_status'] == 'pass'
print('Input checks passed: 92 calendar days, 24 topic-index coverage combinations, and overall validation status = pass.')

Input checks passed: 92 calendar days, 24 topic-index coverage combinations, and overall validation status = pass.


## 1. Market and Download Coverage

In [3]:
market_table = pd.DataFrame([
    ('Target market', quality['market_verification']['question']),
    ('Market ID', quality['market_verification']['market_id']),
    ('Yes token verified', 'Yes'),
    ('Download segments', f"{quality['successful_or_cached_segments']}/{quality['requested_segments']} complete"),
    ('In-range hourly observations', f"{quality['hourly_rows']:,}"),
    ('Out-of-range current-price records removed', quality['outside_segment_count']),
    ('Valid New York calendar days', f"{daily['analysis_p_close'].notna().sum()}/92"),
    ('Stale days', int(daily['stale'].sum())),
    ('2026-07-10 daily close', daily.loc[daily['date'].eq(pd.Timestamp('2026-07-10')), 'p_close'].iloc[0]),
    ('2026-10-09 daily close', daily.loc[daily['date'].eq(pd.Timestamp('2026-10-09')), 'p_close'].iloc[0]),
    ('Window minimum / maximum', f"{daily['p_close'].min():.1%} / {daily['p_close'].max():.1%}"),
], columns=['Item', 'Result'])
display(market_table.style.hide(axis='index').set_properties(**{'text-align': 'left'}).set_table_styles([
    {'selector': 'th', 'props': [('background-color', '#1f4e78'), ('color', 'white'), ('font-weight', 'bold')]},
]))

Item,Result
Target market,"2026 Balance of Power: D Senate, D House"
Market ID,562828
Yes token verified,Yes
Download segments,14/14 complete
In-range hourly observations,"2,205"
Out-of-range current-price records removed,14
Valid New York calendar days,92/92
Stale days,0
2026-07-10 daily close,0.445000
2026-10-09 daily close,0.615000


## 2. Valid and Missing Days by Topic and Index

The market-price series is valid on all 92 days, so missing values here come from the existing Guardian indices. Explicitly neutral observations remain valid zeros; no expressed stance, no relevant article, or an unclear judgment remains missing.

In [4]:
coverage_table = coverage.copy()
coverage_table['Index'] = coverage_table['index_type'].map(INDEX_NAMES)
coverage_table['Topic'] = coverage_table['topic_id'].map(lambda x: f'T{x} {TOPIC_NAMES[x]}')
coverage_table = coverage_table[[
    'Index', 'Topic', 'index_observed_days', 'index_missing_days',
    'joint_adjacent_change_days', 'low_sample_days'
]].rename(columns={
    'index_observed_days': 'Valid index days',
    'index_missing_days': 'Missing days',
    'joint_adjacent_change_days': 'Adjacent-change pairs (N)',
    'low_sample_days': 'Low-sample days',
})
display(coverage_table.style.hide(axis='index').set_table_styles([
    {'selector': 'th', 'props': [('background-color', '#334155'), ('color', 'white')]},
]).set_properties(subset=['Valid index days', 'Missing days', 'Adjacent-change pairs (N)', 'Low-sample days'], **{'text-align': 'right'}))

Index,Topic,Valid index days,Missing days,Adjacent-change pairs (N),Low-sample days
Democratic stance,"T0 AI, Datacenters, and Digital Governance",11,81,1,92
Democratic stance,"T1 Infrastructure, Public Lands, and Federal Investment",14,78,2,92
Democratic stance,"T2 Trade, Public Health, and Legal Rights",48,44,30,88
Democratic stance,"T3 Iran War, Oil, and Energy Security",16,76,6,91
Democratic stance,"T4 Immigration, Public Funding, and Household Economy",35,57,16,91
Democratic stance,T5 Defense Spending and Military Security,22,70,5,89
Republican stance,"T0 AI, Datacenters, and Digital Governance",12,80,3,92
Republican stance,"T1 Infrastructure, Public Lands, and Federal Investment",19,73,4,92
Republican stance,"T2 Trade, Public Health, and Legal Rights",63,29,46,82
Republican stance,"T3 Iran War, Oil, and Energy Security",40,52,21,89


## 3. Primary Test: Same-Day Change Correlations

The table reports the pre-specified Pearson `corr(ΔS_t, ΔP_t)`. Confidence intervals use 1,000 moving-block bootstrap repetitions with seven-day blocks. BH q-values use the pre-defined family of 36 primary tests (six topics × three sentiment/party indices × Pearson/Spearman).

In [5]:
main_corr = corr.loc[
    corr['primary_test'].eq(True) & corr['method'].eq('pearson')
].copy()
main_corr['Index'] = main_corr['index_type'].map(INDEX_NAMES)
main_corr['Topic'] = main_corr['topic_id'].map(lambda x: f'T{x} {TOPIC_NAMES[x]}')
main_corr['95% block CI'] = main_corr.apply(
    lambda r: 'NA' if pd.isna(r['bootstrap_ci_low']) else f"[{r['bootstrap_ci_low']:.3f}, {r['bootstrap_ci_high']:.3f}]", axis=1
)
main_corr['BH significant'] = main_corr['bh_reject_0_05'].map({True: 'Yes', False: 'No'})
main_corr['Sample-size flag'] = main_corr['inference_reliable_n'].map({True: 'N≥20', False: 'N<20'})
main_corr_table = main_corr[[
    'Index', 'Topic', 'coefficient', 'effective_n', '95% block CI',
    'block_null_p', 'bh_q', 'BH significant', 'Sample-size flag'
]].rename(columns={
    'coefficient': 'Pearson r', 'effective_n': 'Effective N',
    'block_null_p': 'Block-null p', 'bh_q': 'BH q'
})

def corr_css(value):
    if pd.isna(value):
        return 'background-color: #eeeeee; color: #666666'
    intensity = min(abs(float(value)), 1.0)
    return f'background-color: rgba({220 if value < 0 else 80}, {100 if value < 0 else 140}, {100 if value < 0 else 220}, {0.12 + 0.45 * intensity})'

display(main_corr_table.style.hide(axis='index')
    .format({'Pearson r': '{:+.3f}', 'Block-null p': '{:.3f}', 'BH q': '{:.3f}'}, na_rep='NA')
    .map(corr_css, subset=['Pearson r'])
    .set_table_styles([{'selector': 'th', 'props': [('background-color', '#1f4e78'), ('color', 'white')]}]))
print(f"BH-adjusted significance: {int(main_corr['bh_reject_0_05'].sum())}/{len(main_corr)} Pearson primary tests; the full family including Spearman also has zero significant results.")

Index,Topic,Pearson r,Effective N,95% block CI,Block-null p,BH q,BH significant,Sample-size flag
Democratic stance,"T0 AI, Datacenters, and Digital Governance",NA,1,NA,NA,NA,No,N<20
Democratic stance,"T1 Infrastructure, Public Lands, and Federal Investment",NA,2,NA,NA,NA,No,N<20
Democratic stance,"T2 Trade, Public Health, and Legal Rights",+0.191,30,"[0.000, 0.428]",0.282,0.608,No,N≥20
Democratic stance,"T3 Iran War, Oil, and Energy Security",+0.000,6,"[-0.270, 0.343]",1.000,1.000,No,N<20
Democratic stance,"T4 Immigration, Public Funding, and Household Economy",+0.049,16,"[-0.228, 0.231]",0.835,1.000,No,N<20
Democratic stance,T5 Defense Spending and Military Security,+0.000,5,"[-0.478, 0.559]",1.000,1.000,No,N<20
Republican stance,"T0 AI, Datacenters, and Digital Governance",NA,3,NA,NA,NA,No,N<20
Republican stance,"T1 Infrastructure, Public Lands, and Federal Investment",NA,4,NA,NA,NA,No,N<20
Republican stance,"T2 Trade, Public Health, and Legal Rights",+0.306,46,"[-0.132, 0.583]",0.040,0.587,No,N≥20
Republican stance,"T3 Iran War, Oil, and Energy Security",+0.302,21,"[-0.095, 0.740]",0.183,0.587,No,N≥20


BH-adjusted significance: 0/18 Pearson primary tests; the full family including Spearman also has zero significant results.


## 4. Expanding-Window Out-of-Sample Prediction

Each prediction requires at least 30 training pairs whose outcomes were already known. Models and benchmarks are compared on the same valid prediction dates. Actual zero changes are excluded from the directional-accuracy denominator.

In [6]:
completed = metrics.loc[metrics['status'].eq('completed')].copy()
completed['Index'] = completed['index_type'].map(INDEX_NAMES)
completed['Topic'] = completed['topic_id'].map(lambda x: f'T{x} {TOPIC_NAMES[x]}')
completed['Feature'] = completed['feature_variant'].map({'level': 'Level S_t', 'change': 'Change ΔS_t'})
completed['Beats both benchmarks'] = (completed['beats_zero_rmse'] & completed['beats_train_mean_rmse']).map({True: 'Yes', False: 'No'})
oos_table = completed[[
    'Index', 'Topic', 'Feature', 'prediction_count', 'model_rmse', 'zero_rmse',
    'train_mean_rmse', 'oos_r2_vs_zero', 'oos_r2_vs_train_mean',
    'direction_accuracy_excluding_actual_zeros', 'direction_denominator', 'Beats both benchmarks'
]].rename(columns={
    'prediction_count': 'Prediction N', 'model_rmse': 'Model RMSE',
    'zero_rmse': 'Zero-change RMSE', 'train_mean_rmse': 'Training-mean RMSE',
    'oos_r2_vs_zero': 'OOS R² vs zero', 'oos_r2_vs_train_mean': 'OOS R² vs mean',
    'direction_accuracy_excluding_actual_zeros': 'Directional accuracy',
    'direction_denominator': 'Direction N',
})
display(oos_table.style.hide(axis='index').format({
    'Model RMSE': '{:.4f}', 'Zero-change RMSE': '{:.4f}', 'Training-mean RMSE': '{:.4f}',
    'OOS R² vs zero': '{:+.3f}', 'OOS R² vs mean': '{:+.3f}', 'Directional accuracy': '{:.1%}',
}, na_rep='NA').set_table_styles([
    {'selector': 'th', 'props': [('background-color', '#334155'), ('color', 'white')]},
]))

Index,Topic,Feature,Prediction N,Model RMSE,Zero-change RMSE,Training-mean RMSE,OOS R² vs zero,OOS R² vs mean,Directional accuracy,Direction N,Beats both benchmarks
Democratic stance,"T2 Trade, Public Health, and Legal Rights",Level S_t,18,0.0131,0.0137,0.0133,+0.091,+0.028,70.0%,10.000000,Yes
Democratic stance,"T4 Immigration, Public Funding, and Household Economy",Level S_t,5,0.0095,0.0089,0.0094,-0.121,-0.011,50.0%,4.000000,No
Republican stance,"T2 Trade, Public Health, and Legal Rights",Level S_t,33,0.0142,0.0146,0.0141,+0.051,-0.014,75.0%,16.000000,No
Republican stance,"T2 Trade, Public Health, and Legal Rights",Change ΔS_t,16,0.0155,0.0158,0.0157,+0.034,+0.018,60.0%,10.000000,Yes
Republican stance,"T3 Iran War, Oil, and Energy Security",Level S_t,10,0.0136,0.0134,0.0137,-0.034,+0.012,42.9%,7.000000,No
Republican stance,"T4 Immigration, Public Funding, and Household Economy",Level S_t,17,0.0166,0.0159,0.0159,-0.087,-0.089,45.5%,11.000000,No
Republican stance,T5 Defense Spending and Military Security,Level S_t,8,0.0136,0.0127,0.0135,-0.143,-0.015,60.0%,5.000000,No
Topic sentiment,"T2 Trade, Public Health, and Legal Rights",Level S_t,44,0.0141,0.0142,0.0140,+0.020,-0.011,62.5%,24.000000,No
Topic sentiment,"T2 Trade, Public Health, and Legal Rights",Change ΔS_t,31,0.0133,0.0134,0.0133,+0.019,+0.005,50.0%,18.000000,Yes
Topic sentiment,"T3 Iran War, Oil, and Energy Security",Level S_t,26,0.0153,0.0152,0.0153,-0.012,+0.004,57.1%,14.000000,No


### Models That Beat Both the Zero-Change and Training-Mean Benchmarks

These results come from the fixed model set rather than ex-post retuning. Evaluation samples remain small and improvements are limited.

In [7]:
beat_both = oos_table.loc[oos_table['Beats both benchmarks'].eq('Yes')].copy()
display(beat_both.style.hide(axis='index').format({
    'Model RMSE': '{:.4f}', 'Zero-change RMSE': '{:.4f}', 'Training-mean RMSE': '{:.4f}',
    'OOS R² vs zero': '{:+.3f}', 'OOS R² vs mean': '{:+.3f}', 'Directional accuracy': '{:.1%}',
}, na_rep='NA').set_table_styles([
    {'selector': 'th', 'props': [('background-color', '#166534'), ('color', 'white')]},
]))
print(f'{len(beat_both)} models beat both benchmarks; this does not establish tradable profitability.')

Index,Topic,Feature,Prediction N,Model RMSE,Zero-change RMSE,Training-mean RMSE,OOS R² vs zero,OOS R² vs mean,Directional accuracy,Direction N,Beats both benchmarks
Democratic stance,"T2 Trade, Public Health, and Legal Rights",Level S_t,18,0.0131,0.0137,0.0133,+0.091,+0.028,70.0%,10.000000,Yes
Republican stance,"T2 Trade, Public Health, and Legal Rights",Change ΔS_t,16,0.0155,0.0158,0.0157,+0.034,+0.018,60.0%,10.000000,Yes
Topic sentiment,"T2 Trade, Public Health, and Legal Rights",Change ΔS_t,31,0.0133,0.0134,0.0133,+0.019,+0.005,50.0%,18.000000,Yes
Topic sentiment,"T4 Immigration, Public Funding, and Household Economy",Level S_t,32,0.0154,0.0155,0.0154,+0.020,+0.006,55.6%,18.000000,Yes


4 models beat both benchmarks; this does not establish tradable profitability.


## 5. Validation Results

In [8]:
validation_rows = []
for name, value in validation['checks'].items():
    if isinstance(value, bool):
        shown = 'Pass' if value else 'Fail'
    elif isinstance(value, dict):
        shown = value.get('status', str(value))
    else:
        shown = value
    validation_rows.append({'Check': name, 'Result': shown})
validation_table = pd.DataFrame(validation_rows)
display(validation_table.style.hide(axis='index').set_table_styles([
    {'selector': 'th', 'props': [('background-color', '#475569'), ('color', 'white')]},
]))
print('Overall validation status:', validation['validation_status'])

Check,Result
market_id_matches,Pass
yes_token_matches,Pass
hourly_rows_in_requested_range,Pass
hourly_probabilities_in_range,Pass
hourly_timestamps_unique,Pass
daily_has_92_rows,Pass
aligned_unique_key,Pass
aligned_expected_rows,Pass
no_diff_across_missing_calendar_days,Pass
walk_forward_is_causal,Pass


Overall validation status: pass


## Conclusion

- None of the primary same-day change tests is significant after BH adjustment.
- T0, T1, and several party-stance series are too sparse for reliable inference.
- Only 4 of the 15 evaluable out-of-sample models beat both simple benchmarks, and the improvements are small.
- These are exploratory predictive tests, not causal conclusions or trading-return backtests.